# Final Model Training & Hybrid Ensemble

This notebook combines our optimized Classical Machine Learning models (LGBM, XGBoost, CatBoost) with our specialized 5-Channel DistilBERT deep learning model to generate the final competition predictions.

## Data Acquisition
The dataset is retrieved directly from the Kaggle competition using the Kaggle API.

To reproduce this environment:
1. Upload your `kaggle.json` API token.
2. Run the following commands to download and extract the data:

```bash
# !pip install kaggle
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c jigsaw-agile-community-rules
# !unzip jigsaw-agile-community-rules.zip -d data

## 1. Imports and GPU Configuration

In [10]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import gc
import json
import joblib
import scipy.sparse as sp
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
#!pip install catboost
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier
from huggingface_hub import hf_hub_download

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test = pd.read_csv("data/test.csv")

## 2. Load Artifacts and Hyperparameters
In this section, we load the pre-processed sparse matrices, training labels, and the optimized hyperparameters discovered during the Optuna tuning phase.

In [12]:
# Load Optimized Hyperparameters
with open('best_hyperparameters.json', 'r') as f:
    best_params = json.load(f)

# Load Pre-processed Data
X_train_csr = sp.load_npz(os.path.join('X_train_csr.npz'))
X_test_csr = sp.load_npz(os.path.join('X_test_csr.npz'))
y_train = joblib.load(os.path.join('y_train.pkl'))
test_ids = joblib.load(os.path.join('test_ids.pkl'))

# Load test dataframe for DistilBERT text processing
test = pd.read_csv('data/test.csv')

## 3. Classical ML Ensemble Training (Voting)
We utilize a Soft Voting Classifier to ensemble our Gradient Boosting Machines. This provides a robust fallback and captures structured patterns in the data.

In [13]:
clf1 = LGBMClassifier(**best_params['lgbm'], n_estimators=500, force_col_wise=True)
clf2 = XGBClassifier(**best_params['xgb'], n_estimators=500, tree_method='hist', device='cpu')
clf3 = CatBoostClassifier(**best_params['cat'], iterations=500, task_type='CPU', verbose=0)

voting_clf = VotingClassifier(
    estimators=[('lgbm', clf1), ('xgb', clf2), ('cat', clf3)],
    voting='soft'
)

# Training on the full dataset (X_train_csr and y_train should be loaded)
voting_clf.fit(X_train_csr, y_train.values if hasattr(y_train, 'values') else y_train)

[LightGBM] [Info] Number of positive: 1031, number of negative: 998
[LightGBM] [Info] Total Bins 1000045
[LightGBM] [Info] Number of data points in the train set: 2029, number of used features: 7001
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.508132 -> initscore=0.032531
[LightGBM] [Info] Start training from score 0.032531
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

VotingClassifier(estimators=[('lgbm',
                              LGBMClassifier(colsample_bytree=0.8704872244974341,
                                             force_col_wise=True,
                                             learning_rate=0.05788536024938373,
                                             max_depth=12, n_estimators=500,
                                             num_leaves=90,
                                             reg_alpha=0.0060150916672568645,
                                             reg_lambda=0.003449486887507533,
                                             subsample=0.8659995717946736)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            cols...
                                            learning_rate=0.018074966193584112,
                                            max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=5,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=500, n_jobs=None,
                                            num_parallel_tree=None, ...)),
                             ('cat',
                              <catboost.core.CatBoostClassifier object at 0x7f7beeb8f980>)],
                 voting='soft')

## 4. Classical Model Inference
Processing test data in batches to optimize memory usage.

In [14]:
test_classic_probs = []
batch_size_classic = 1000

for i in range(0, X_test_csr.shape[0], batch_size_classic):
    batch_probs = voting_clf.predict_proba(X_test_csr[i:i+batch_size_classic])[:, 1]
    test_classic_probs.append(batch_probs)

test_classic_probs = np.concatenate(test_classic_probs)
gc.collect()

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


42

## 5. 5-Channel DistilBERT Architecture
This custom architecture designed to process rule violations by analyzing the main body text alongside positive and negative examples simultaneously.

In [17]:
class FiveChannelDistilBert(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.distilbert = AutoModel.from_pretrained(model_path)
        self.classifier = nn.Linear(5, 1)

    def get_score(self, inputs):
        input_ids = inputs['input_ids'].squeeze(1).to(device)
        attention_mask = inputs['attention_mask'].squeeze(1).to(device)
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        # Using the [CLS] token representation (index 0)
        return torch.mean(outputs.last_hidden_state[:, 0, :], dim=1)

    def forward(self, batch):
        s_b = self.get_score(batch['body'])
        s_p1 = self.get_score(batch['pos1'])
        s_p2 = self.get_score(batch['pos2'])
        s_n1 = self.get_score(batch['neg1'])
        s_n2 = self.get_score(batch['neg2'])
        combined = torch.stack([s_b, s_p1, s_p2, s_n1, s_n2], dim=1)
        return self.classifier(combined).squeeze(-1)

## 6. DistilBERT Inference
Loading the pre-trained weights and performing a forward pass on the test set.

In [18]:
def get_distilbert_preds(test_df, repo_id, filename):
    local_weights_path = hf_hub_download(repo_id=repo_id, filename=filename)

    tokenizer = AutoTokenizer.from_pretrained(repo_id)
    model = FiveChannelDistilBert(repo_id)

    model.load_state_dict(torch.load(local_weights_path, map_location=device))
    model.to(device)
    model.eval()

    all_preds = []
    batch_size = 8

    with torch.no_grad():
        for i in tqdm(range(0, len(test_df), batch_size)):
            batch_df = test_df.iloc[i : i + batch_size]
            rules = batch_df['rule'].astype(str).values

            def tokenize_texts(texts):
                return tokenizer(list(texts), rules.tolist(), padding='max_length',
                                 truncation=True, max_length=128, return_tensors="pt").to(device)

            batch_inputs = {
                "body": tokenize_texts(batch_df['body'].values),
                "pos1": tokenize_texts(batch_df['positive_example_1'].values),
                "pos2": tokenize_texts(batch_df['positive_example_2'].values),
                "neg1": tokenize_texts(batch_df['negative_example_1'].values),
                "neg2": tokenize_texts(batch_df['negative_example_2'].values)
            }

            logits = model(batch_inputs)
            preds = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(preds)

            del batch_inputs, logits
            torch.cuda.empty_cache()

    return np.concatenate(all_preds)

# Predictions Configuration
REPO_ID = "BernaTS/distilbert-5channel-jigsaw"
FILENAME = "five_channel_final_weights.pt"

test_dbert_probs = get_distilbert_preds(test, REPO_ID, FILENAME)

test_classic_probs = []
for i in range(0, X_test_csr.shape[0], 1000):
    test_classic_probs.append(voting_clf.predict_proba(X_test_csr[i:i+1000])[:, 1])

test_classic_probs = np.concatenate(test_classic_probs)

model.safetensors:   0%|          | 0.00/265M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00,  3.03it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 7. Hybrid Ensemble
We combine the deep learning predictions (75% weight) with the classical ensemble predictions (25% weight) to produce the final results.

In [19]:
W_DBERT = 0.75
W_CLASSIC = 0.25

# Weighted average blending
final_test_probs = (test_dbert_probs * W_DBERT) + (test_classic_probs * W_CLASSIC)

## 8. Final Submission

In [20]:
submission = pd.DataFrame({
    'row_id': test['row_id'],
    'rule_violation': final_test_probs
})

submission.to_csv('submission.csv', index=False)

## 9. Local Download (Optional)

submission.csv file can be downloaded using the following code block.

```bash
# from google.colab import files
# files.download('submission.csv')
